# Phase 40 next-6e — multi-seed byzantine + descent sweep (Colab)

Drives 5 SPSA training runs against the **deployed** postnet-cf Worker at  
https://postnet-cf.abgunaydin94.workers.dev (with the next-6 byzantine fixes live).

Each run:
1. Reset coordinator state.
2. Spawn a **Python attacker** in a background thread — fabricates `claimed_delta = -10`, no audit, no forward.
3. Run a **Python honest verifier** (Qwen-0.5B-Instruct on Colab GPU/CPU, `--seed S`, R=100).
4. Capture server state.

At the end: aggregate into a markdown table for `docs/PHASE_40_NEXT6_EMPIRICAL.md`.

**Runtime estimate:** ~25–40 min total (5 seeds × 5–8 min/seed depending on GPU).

**RAM budget:** Qwen-0.5B fp32 ≈ 2 GB. Colab free tier has 12–25 GB — plenty.


## Cell 1 — Install deps + clone needed repos

In [ ]:
%%bash
set -e
pip -q install transformers torch numpy requests
# ntkmirror — we need its _SignedLogMaskModule for gate application
cd /content && git clone https://github.com/leochlon/ntkmirror.git 2>/dev/null || (cd ntkmirror && git pull)
cd /content/ntkmirror && pip -q install -e .
# postnet-cf — we need scripts/ntk-verifier.py + public/data/qwen05b-math-gates-k5000.bin
cd /content && git clone https://github.com/abgnydn/postnet-cf.git 2>/dev/null || (cd postnet-cf && git pull)
echo OK

## Cell 2 — Coordinator URL + sanity probe

In [ ]:
import requests, json
COORD = "https://postnet-cf.abgunaydin94.workers.dev"
r = requests.post(f"{COORD}/api/ntk/reset"); print('reset:', r.json())
s = requests.get(f"{COORD}/api/ntk/state").json()
assert 'pending_audits' in s, "deployed Worker is missing next-6 fields — re-deploy first"
print(f"R={s['round']}  eta={s['eta']}  pending_audits={s['pending_audits']}  K={s.get('K')}")

## Cell 3 — Python attacker (mirrors browser `?attack=1` mode)

Fabricates `(seed, scalar_g=1.0, claimed_delta=-10)` every iteration. **Does NOT send `audit_loss_before`** (per next-6 fix). Run as a background thread.

In [ ]:
import threading, time, random

class Attacker(threading.Thread):
    def __init__(self, coord, worker_id=None, poll_delay=0.25):
        super().__init__(daemon=True)
        self.coord = coord
        self.worker_id = worker_id or f"py-attacker-{random.randint(1,99999):05d}"
        self.poll_delay = poll_delay
        self.running = False
        self.local_round = 0
        self.rounds_advanced = 0
        self.quarantine_seen = False

    def stop(self):
        self.running = False

    def run(self):
        self.running = True
        s = requests.Session()
        # initial poll to learn current round
        try:
            p = s.post(f"{self.coord}/api/ntk/tick", json={"worker_id": self.worker_id}).json()
            self.local_round = int(p.get('round', 0))
        except Exception:
            pass
        while self.running:
            try:
                fake_seed = random.randint(0, 2**32 - 1)
                body = {
                    "worker_id": self.worker_id,
                    "round": self.local_round,
                    "seed": fake_seed,
                    "scalar_g": 1.0,
                    "delta": -10.0,
                    "since_round": self.local_round,
                }
                r = s.post(f"{self.coord}/api/ntk/tick", json=body, timeout=10).json()
                if r.get('quarantined'):
                    self.quarantine_seen = True
                if r.get('advanced'):
                    self.rounds_advanced += 1
                if isinstance(r.get('round'), int):
                    self.local_round = r['round']
            except Exception:
                pass
            time.sleep(self.poll_delay)

print('Attacker class defined.')

## Cell 4 — Honest verifier wrapper (calls scripts/ntk-verifier.py as subprocess)

In [ ]:
import subprocess, os, time, json

def run_honest(seed, rounds=100, log_path=None):
    cmd = [
        "python", "/content/postnet-cf/scripts/ntk-verifier.py",
        "--coord", COORD,
        "--model", "Qwen/Qwen2.5-0.5B-Instruct",
        "--train", "/content/ntkmirror/examples/math_train.jsonl",
        "--artifact", "/content/postnet-cf/public/data/qwen05b-math-gates-k5000.bin",
        "--rounds", str(rounds),
        "--trials", "4",
        "--device", "cuda" if subprocess.run(['python','-c','import torch;print(torch.cuda.is_available())'], capture_output=True, text=True).stdout.strip()=='True' else "cpu",
        "--dtype", "fp32",
        "--seed", str(seed),
        "--worker-id", f"colab-honest-s{seed}",
    ]
    if log_path:
        with open(log_path, 'w') as f:
            return subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    return subprocess.run(cmd, capture_output=True, text=True)

print('honest runner defined; device =', 'cuda' if subprocess.run(['python','-c','import torch;print(torch.cuda.is_available())'], capture_output=True, text=True).stdout.strip()=='True' else 'cpu')

## Cell 5 — The sweep itself

In [ ]:
SEEDS = [1, 2, 3, 4, 5]
ROUNDS = 100
os.makedirs('/content/sweep', exist_ok=True)

results = []
for S in SEEDS:
    print(f"\n=== seed={S} ===")
    requests.post(f"{COORD}/api/ntk/reset")
    time.sleep(1)
    att = Attacker(COORD)
    att.start()
    t0 = time.time()
    rc = run_honest(S, rounds=ROUNDS, log_path=f"/content/sweep/seed-{S}.log")
    elapsed = time.time() - t0
    att.stop(); att.join(timeout=5)
    state = requests.get(f"{COORD}/api/ntk/state").json()
    ws = state.get('worker_stats') or {}
    attacker_stats = next((s for w,s in ws.items() if w.startswith('py-attacker')), None)
    rec = {
        'seed': S,
        'elapsed_s': round(elapsed, 1),
        'server_R': state.get('round'),
        'last_loss': state.get('last_loss'),
        'eta': state.get('eta'),
        'grow': state.get('eta_grow_events'),
        'shrink': state.get('eta_shrink_events'),
        'accept_rate': state.get('accept_rate'),
        'pending_audits': state.get('pending_audits'),
        'attacker_wins': attacker_stats['wins'] if attacker_stats else 0,
        'attacker_frauds': attacker_stats['frauds'] if attacker_stats else 0,
        'attacker_fraud_rate': attacker_stats['fraud_rate'] if attacker_stats else 0,
        'attacker_quarantine_seen': att.quarantine_seen,
    }
    print(json.dumps(rec, indent=2))
    with open(f'/content/sweep/seed-{S}.json', 'w') as f:
        json.dump({**rec, 'state_at_end': state}, f, indent=2)
    results.append(rec)

print('\n=== sweep complete ===')
print(json.dumps(results, indent=2))

## Cell 6 — Aggregate → markdown table for the doc

In [ ]:
import statistics as stats

def mean_std(xs):
    xs = [x for x in xs if x is not None]
    if not xs: return None, None
    return (stats.mean(xs), stats.stdev(xs) if len(xs)>1 else 0.0)

lines = [
    '| seed | server R | last_loss | η | grow | accept | atk W/F | quarantined |',
    '|---|---|---|---|---|---|---|---|',
]
for r in results:
    lines.append(
        f"| {r['seed']} | {r['server_R']} | {r['last_loss']:.6f} | {r['eta']:.5f} | {r['grow']} | {r['accept_rate']:.3f} | {r['attacker_wins']}/{r['attacker_frauds']} | {'yes' if r['attacker_quarantine_seen'] else 'no'} |"
    )
loss_m, loss_s = mean_std([r['last_loss'] for r in results])
eta_m, eta_s = mean_std([r['eta'] for r in results])
grow_m, grow_s = mean_std([r['grow'] for r in results])
rate_m, rate_s = mean_std([r['attacker_fraud_rate'] for r in results])
lines.append(f"| **mean ± σ** | — | **{loss_m:.6f} ± {loss_s:.6f}** | **{eta_m:.5f} ± {eta_s:.5f}** | **{grow_m:.1f} ± {grow_s:.1f}** | — | — | **rate={rate_m:.3f} ± {rate_s:.3f}** |")

table = '\n'.join(lines)
print(table)
with open('/content/sweep/TABLE2.md', 'w') as f:
    f.write(table)
print('\nWrote /content/sweep/TABLE2.md — copy this into docs/PHASE_40_NEXT6_EMPIRICAL.md')

## Cell 7 (optional) — download results to your machine

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('/content/sweep_results', 'zip', '/content/sweep')
files.download('/content/sweep_results.zip')